In [1]:
import csv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

def search_videos_by_keyword(keyword, api_key, max_videos=10):
    """Mencari video YouTube berdasarkan kata kunci tertentu"""
    try:
        youtube = build('youtube', 'v3', developerKey=api_key)

        print(f"Mencari video dengan kata kunci: '{keyword}'...")
        search_response = youtube.search().list(
            q=keyword,
            part='id,snippet',
            type='video',
            maxResults=max_videos,
            regionCode='ID', # Fokus region Indonesia
            relevanceLanguage='id'
        ).execute()

        video_list = []
        for item in search_response.get('items', []):
            video_id = item['id']['videoId']
            video_title = item['snippet']['title']
            video_list.append({'video_id': video_id, 'title': video_title})

        print(f"Ditemukan {len(video_list)} video.")
        return video_list

    except HttpError as e:
        print(f'Error saat mencari video: {e}')
        return []

def get_all_comments(video_id, api_key):
    try:
        youtube = build('youtube', 'v3', developerKey=api_key)
        comments = []
        next_page_token = None

        while True:
            response = youtube.commentThreads().list(
                part='snippet',
                videoId=video_id,
                maxResults=100,
                pageToken=next_page_token
            ).execute()

            for item in response['items']:
                comment_id = item['id']
                comment_snippet = item['snippet']['topLevelComment']['snippet']

                author_name = comment_snippet.get('authorDisplayName', '')
                author_id = comment_snippet.get('authorChannelId', {}).get('value', '')
                comment_text = comment_snippet.get('textDisplay', '')
                like_count = comment_snippet.get('likeCount', 0)
                comment_link = f"https://www.youtube.com/watch?v={video_id}&lc={comment_id}"

                comments.append({
                    'Video ID': video_id,
                    'Comment ID': comment_id,
                    'Comment Link': comment_link,
                    'Commenter Name': author_name,
                    'Commenter ID': author_id,
                    'Comment': comment_text,
                    'Likes': like_count
                })

            next_page_token = response.get('nextPageToken')
            if not next_page_token:
                break

        return comments
    except HttpError as e:
        print(f'Error ambil komentar dari video {video_id}: {e}')
        return []

api_key = '' #Enter your API KEY
keyword_pencarian = 'Rupiah melemah'

# 1. Cari videonya
videos = search_videos_by_keyword(keyword_pencarian, api_key, max_videos=100)

all_combined_comments = []

# 2. Ambil komentar dari tiap video
for vid in videos:
    print(f"\nMengambil komentar dari video: {vid['title']} (ID: {vid['video_id']})")
    comments = get_all_comments(vid['video_id'], api_key)
    all_combined_comments.extend(comments)

# 3. Simpan ke CSV gabungan
if all_combined_comments:
    filename = 'rupiah_melemah_comments.csv'
    fieldnames = ['Video ID', 'Comment ID', 'Comment Link', 'Commenter Name', 'Commenter ID', 'Comment', 'Likes']

    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        for comment in all_combined_comments:
            writer.writerow(comment)

    print(f"\nSukses! Total {len(all_combined_comments)} komentar dari beberapa video telah disimpan ke '{filename}'.")
else:
    print("Tidak ada data komentar yang berhasil dikumpulkan.")

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.